# Data quality and idempotency validation
This notebook acts as the final validation gate for the pipeline. It checks row counts and primary key uniqueness across all six tables in Bronze, Silver, and Gold, verifies that no `ERROR`-level checks failed in `ops.data_quality_results`, and displays the final Gold table.

### Key design choices in this notebook
* **End-to-end uniqueness verification:** Loops through all six physical Delta tables and checks their business keys (`bronze_record_id`, `(price_area, time_start_utc)`, `(station_id, time_start_utc)`, `(price_area, station_id, time_start_utc)`, and `(price_area, temperature_band_order)`) to confirm `duplicate_key_count == 0`. This proves that our Delta `MERGE` logic is idempotent even after re-runs.
* **Centralized quality gate (`ops.data_quality_results`):** Queries the operational audit table and fails the workflow with an `AssertionError` if any check with `severity == 'ERROR'` has `passed == False`, while allowing `WARNING` checks (such as `weather_match_rate` or `unusual_weather_values`) to remain visible for review.
* **Blocking downstream tasks:** In the Databricks Workflow DAG, this task runs after `05_gold_analysis` and before the ML and Gen AI tasks so downstream models only run when the core Medallion tables pass all checks.

In [0]:
from pyspark.sql import functions as F

# Lock Spark session to UTC and dynamically resolve the active Unity Catalog
spark.conf.set("spark.sql.session.timeZone", "UTC")
CATALOG = spark.sql("SELECT current_catalog()").first()[0]

### Verify row counts and key uniqueness across all layers
Checks each table in Bronze, Silver, and Gold to confirm that `total_rows > 0` and `duplicate_key_count == 0` for its business key. Raises an `AssertionError` if any table is empty or contains duplicate keys.


In [0]:
# Define each Medallion table and its expected unique business key columns
tables_to_check = [
    ("bronze.electricity_price_api_raw", ["bronze_record_id"]),
    ("bronze.weather_api_raw", ["bronze_record_id"]),
    ("silver.electricity_price_hourly", ["price_area", "time_start_utc"]),
    ("silver.weather_hourly", ["station_id", "time_start_utc"]),
    ("silver.weather_price_hourly", ["price_area", "station_id", "time_start_utc"]),
    ("gold.price_by_temperature_band", ["price_area", "temperature_band_order"])
]

# Count total rows and check for any duplicate business keys in each table
summary_rows = []
for table_path, key_cols in tables_to_check:
    full_name = f"{CATALOG}.{table_path}"
    df = spark.table(full_name)
    total_rows = df.count()
    duplicate_keys = (
        df.groupBy(*key_cols).count().filter("count > 1").count()
    )
    summary_rows.append((table_path, total_rows, duplicate_keys))
    
    # Fail the validation gate if a table is empty or has duplicate primary keys
    if total_rows == 0:
        raise AssertionError(f"Validation failed: {full_name} is empty")
    if duplicate_keys > 0:
        raise AssertionError(
            f"Validation failed: {full_name} has {duplicate_keys} duplicate key(s)"
        )

# Display the summary table showing row counts and 0 duplicates across all layers
summary_df = spark.createDataFrame(
    summary_rows, ["table_name", "total_rows", "duplicate_key_count"]
)
display(summary_df)

### Check operational data quality logs
Reads `ops.data_quality_results`, displays the 20 most recent check entries across all layers, and raises an `AssertionError` if any `ERROR`-level check failed.

In [0]:
# Load the operational data quality log table and count any failed ERROR checks
dq_df = spark.table(f"{CATALOG}.ops.data_quality_results")
failed_errors = dq_df.filter(
    (F.col("severity") == "ERROR") & (F.col("passed") == False)
).count()

# Display the 20 most recent data quality check results for inspection
display(
    dq_df.orderBy(F.col("checked_at_utc").desc()).limit(20)
)

# Stop the workflow if any mandatory ERROR check failed
if failed_errors > 0:
    raise AssertionError(
        f"Validation failed: {failed_errors} ERROR-level data quality check(s) failed"
    )

print("All table uniqueness and data quality checks passed")

### Display final Gold output and chart
Queries `gold.price_by_temperature_band` ordered from coldest to warmest via `temperature_band_order` and visualizes the median and average electricity price (`NOK/kWh`) across temperature bands.

In [0]:
# Load the Gold table ordered from coldest to warmest band for display and charting
gold_df = (
    spark.table(f"{CATALOG}.gold.price_by_temperature_band")
    .orderBy("temperature_band_order")
)
display(gold_df)

Databricks visualization. Run in Databricks to view.